### Key Observations
* The number of job openings at Amazon declined by **91%** between June 2022 and July 2023, falling from 91,205 open positions to just 8,412 across 34 job categories.
* By November 2026, the percentage decline from June 2022 had recovered slightly to **78%**
* Technical roles accounted for a large volume of the decline. For example, in June 2022, there were 32,437 open Software Development positions. In November 2026, the number stood at just 2,582.
* Only Fulfillment roles have shown growth compared with June 2022, as of November 2026.
* See the interactive chart below to inspect changes for individual categories between 2022 and 2023. Updated data for November 2026 is included later.

In [59]:
#| include: false
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow as pa
from pathlib import Path
import ipywidgets as widgets
import plotly.io as pio
import plotly.graph_objects as go
pio.renderers.default = "notebook_connected"


get_ipython().ast_node_interactivity = "all"

In [60]:
#| include: false

CSV_PATH = Path.cwd().parent.parent.parent / "ja_active" / "output" / "counts" / "amz_job_counts_by_category_2022.csv"
counts = pd.read_csv(CSV_PATH, parse_dates=["date"])
counts.columns = ["job_category", "count", "date"]
counts["job_category"] = counts["job_category"].replace("Administrative Support ", "Administrative Support")


In [61]:
#| include: false

## Create a table showing the number of jobs by category at the beginning and end of the period.

def get_job_counts_by_category(counts, date):
    return counts[counts["date"] == date][["job_category", "count"]].sort_values(by="count", ascending=False).reset_index(drop=True).astype({"count": int})

initial_jobs = get_job_counts_by_category(counts, counts["date"].min())
final_jobs = get_job_counts_by_category(counts, counts["date"].max())

def calculate_total_jobs_and_add_row_to_table(counts):
    total_jobs = counts["count"].sum()
    total_row = pd.DataFrame({"job_category": ["Total"], "count": [total_jobs]})
    return pd.concat([counts, total_row], ignore_index=True)

def combine_job_counts(initial, final):
    combined = initial.merge(final, on="job_category", how="outer", suffixes=("_initial", "_final"))
    combined["count_initial"] = combined["count_initial"].fillna(0).astype(int)
    combined["count_final"] = combined["count_final"].fillna(0).astype(int)
    return combined.sort_values(by="count_initial", ascending=False).reset_index(drop=True)


def calculate_difference(combined):
    combined["difference"] = combined["count_final"] - combined["count_initial"]
    return combined

def calculate_difference_percentage(combined):
    combined["difference_percentage"] = round((combined["difference"] / combined["count_initial"].replace(0, 1) * 100), 1)
    return combined

combined_table = combine_job_counts(initial_jobs, final_jobs)
combined_table = calculate_difference(combined_table)
combined_table = calculate_difference_percentage(combined_table)
combined_table.columns = ["Job Category", "June 2022", "July 2023", "Change", "% Change"]



In [62]:
#| echo: false

DEFAULT_CATEGORY = "Software Development"


def build_category_figure(df: pd.DataFrame, job_category: str, date_range="June 2022 - July 2023") -> go.Figure:
    """Build a line chart of open positions over time for one job category."""
    category_data = df.loc[df["job_category"].eq(job_category)].sort_values("date")
    legend_label = (
        f"{job_category}; "
        f"{date_range}"
    )
    fig = go.Figure(
        go.Scatter(
            x=category_data["date"],
            y=category_data["count"],
            mode="lines+markers",
            name=legend_label,
        )
    )
    fig.update_layout(
        xaxis_title="Date",
        yaxis_title="# Open positions",
        showlegend=True,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.04, "x": 0},
        template="plotly_white",
    )
    return fig


def build_category_dropdown_figure(
    df: pd.DataFrame, default_category: str = DEFAULT_CATEGORY, date_range="June 2022 - July 2023"
) -> go.Figure:
    """Build one figure with a trace per category and a dropdown to switch between them."""
    categories = sorted(df["job_category"].dropna().unique())
    fig = go.Figure()
    for category in categories:
        trace = build_category_figure(df, category, date_range=date_range).data[0]
        trace.visible = category == default_category
        fig.add_trace(trace)

    buttons = [
        {
            "label": category,
            "method": "update",
            "args": [{"visible": [c == category for c in categories]}],
        }
        for category in categories
    ]
    fig.update_layout(
        title=dict(
            text="Amazon Open Positions by Job Category",
            x=0.5,
            xanchor="center"
        ),
        xaxis_title="Date",
        yaxis_title="# Open positions",
        showlegend=True,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.05, "x": 0},
        template="plotly_white",
        margin={"t": 140},
        updatemenus=[
            {
                "buttons": buttons,
                "active": categories.index(default_category),
                "direction": "down",
                "x": 0.5,
                "xanchor": "left",
                "y": 0.95,
                "yanchor": "bottom",
            }
        ],
    )
    return fig

build_category_dropdown_figure(counts)

## Data Source and Context

Data was collected on the number of Amazon job openings, by job category, from the [Amazon jobs' website](https://amazon.jobs/content/en/job-categories). Amazon's job openings are advertised under 34 different job categories. The motivation is that Amazon is considered a bellwether of the tech industry.

The data was collected on 243 days between the dates of June 19, 2022 and July 16, 2023, a span of 393 calendar days. 

The results were dramatic. The total number of Amazon job openings fell by **91%** during the period. What follows are summaries of the observations and potential explanations for the dramatic changes in the data.

Data was again collected on the Amazon job openings in the period September 2026 - November 2026. The new data shows that the number of Amazon job openings has not recovered from June 2022. The November 2026 count is higher than in July 2023 but is still **78% lower** than the June 2022 count.


## Overview of the Amazon job categories and counts

The number of job openings per category and the total number of job openings at the beginning and at the end of the period of June 2022 - July 2023 are shown in the table. The table also shows the percentage change in the number of job openings over the period. 

Updated data for November 2026 is added below in the table in section [Open Position Counts in November 2026](#2026-update).

In [63]:
#| echo: false

column_map = {
    "Job Category": "Job Category",
    "June 2022": "June 2022",
    "July 2023": "July 2023",
    "Change": "Change",
    "% Change": "% Change",
}

plotly_table_df = (
    combined_table[list(column_map.keys())]
    .sort_values("% Change", ascending=True)
    .rename(columns=column_map)
    .reset_index(drop=True)
)

fig_table = go.Figure(
    go.Table(
        header=dict(
            values=list(plotly_table_df.columns),
            fill_color="#2c3e50",
            font=dict(color="white", size=12),
            align="center",
        ),
        cells=dict(
            values=[plotly_table_df[col] for col in plotly_table_df.columns],
            fill_color=[["#f7f7f7", "white"] * (len(plotly_table_df) // 2 + 1)],
            align="center",
            height=26,
        ),
    )
)
fig_table.update_layout(
    title=dict(
        text="Amazon Job Counts - July 2023 vs June 2022",
        x=0.5,
        xanchor="center"
    ),
    template="plotly_white",
    height=1000,
    margin={"t": 60, "l": 10, "r": 10, "b": 10},
    
)

The job categories with the largest decreases in open positions were: Software Development (31,802), Solutions Architect (6,465), Project/Program/Product Management combined (9,673) and Operations, IT, & Support Engineering (4,544). 

In percentage terms, Software Development (98%), Public Relations & Communications (98%), Solutions Architect (98%) and Data Science (97%) showed the steepest declines. The overall percentage decline in the number of open positions was 91%.

The job categories with the smallest percentage decline in open positions included: Medical, Health & Safety (23%), Fulfillment & Operations Management (32%), Facilities Maintenance & Real Estate (56%) and Investigation & Loss Prevention (68%).


## Change in Open Positions by Job Category

The following graphs show changes in the period for selected job categories. You can inspect the graphs for any of the individual job categories using the pull down menu.


### Software Development
The number of Software Development positions declined by 98% in the period, higher than the overall decline of 91% for all positions.

In [64]:
#| echo: false
build_category_figure(counts, "Software Development")

### Data Science
The decline in open positions for Data Science is similarly severe. It tracks closely with the decline in Software Development positions in the period.

In [65]:
#| echo: false
build_category_figure(counts, "Data Science")

### Public Relations & Communications
Public Relations & Communications is one of the categories for which the observed declines were very dramatic. The largest declines appear to be concentrated in the period Nov 2022 - January 2023. 

This is consistent with [announcements of hiring embargoes and layoffs by Amazon in the Nov 2022 - Jan 2023 period ](https://www.aboutamazon.com/news/company-news/update-from-ceo-andy-jassy-on-role-eliminations)

In [66]:
#| echo: false
build_category_figure(counts, "Public Relations & Communications")

### Fulfillment & Operations Management
Unlike the Software Development and Data Science categories, Fulfillment & Operations Management showed evidence of a rebound after the minimum was reached in January 2023. Intuitively, this makes some sense, as Amazon still has to fulfill its orders for its retail business to maintain revenues, even if it is reducing costs elsewhere in the business. 

In [67]:
#| echo: false
build_category_figure(counts, "Fulfillment & Operations Management")

### Medical, Health, & Safety
Medical, Health, & Safety showed a particularly strong rebound following the January 2023 minimum and ended the period just 23% below its opening count. This decline is much lower than the overall decline of 91% in open positions during the period. 

In [68]:
#| echo: false
build_category_figure(counts, "Medical, Health, & Safety")

In [69]:
#| include: false

## Read the collected counts for the period September - November 2026

NEW_CSV_PATH = Path.cwd().parent.parent.parent / "ja_active" / "output" / "counts" / "amz_job_counts_by_category_2026.csv"
new_counts = pd.read_csv(NEW_CSV_PATH, parse_dates=["date"])


In [70]:
#| include: false

## Check which categories have been dropped or added
old_categories = set(counts['job_category'].unique())
new_categories = set(new_counts['job_category'].unique())

added_categories = new_categories - old_categories
dropped_categories = old_categories - new_categories

unchanged = old_categories & new_categories

In [71]:
#| include: false

category_mapping = {
    'Audio, Video, & Photography Production': 'Audio / Video / Photography Production',
 'Buying, Planning, & In-Stock Management': 'Buying, Planning, & Instock Management',
 'Design': 'Amazon Design',
 'Finance': 'Finance and Global Business Services',
 'Fulfillment Center Warehouse Associate': 'Fulfillment Associate',
 'Project, Program, & Product Management—Non-Tech': 'Project/Program/Product Management--Non-Tech',
 'Project, Program, & Product Management—Technical': 'Project/Program/Product Management--Technical',
 'Supply Chain & Transportation Management': 'Supply Chain/Transportation Management',
 'Solutions Architecture': 'Solutions Architect',
 "Business Intelligence & Data Engineering": "Business Intelligence",
 "Applied Science": "Machine Learning Science"
}

In [72]:
#| include: false

new_counts["new_job_category"] = new_counts["job_category"].copy()
new_counts["mapped_job_category"] = new_counts["new_job_category"].apply(lambda x: x.replace("and", "&"))
new_counts["mapped_job_category"] = new_counts["mapped_job_category"].astype(str).replace(category_mapping)
new_counts
category_dict = dict(zip(new_counts["job_category"], new_counts["new_job_category"]))


,job_category,count,date,new_job_category,mapped_job_category
0,Administrative Support,663,2026-09-18,Administrative Support,Administrative Support
1,Applied Science,418,2026-09-18,Applied Science,Machine Learning Science
2,"Audio, Video, and Photography Production",34,2026-09-18,"Audio, Video, and Photography Production",Audio / Video / Photography Production
3,Business and Merchant Development,381,2026-09-18,Business and Merchant Development,Business & Merchant Development
4,Business Intelligence and Data Engineering,224,2026-09-18,Business Intelligence and Data Engineering,Business Intelligence
...,...,...,...,...,...
1389,"Sales, Advertising, and Account Management",839,2026-11-05,"Sales, Advertising, and Account Management","Sales, Advertising, & Account Management"
1390,Software Development,2582,2026-11-05,Software Development,Software Development
1391,Solutions Architecture,810,2026-11-05,Solutions Architecture,Solutions Architect
1392,Supply Chain and Transportation Management,543,2026-11-05,Supply Chain and Transportation Management,Supply Chain/Transportation Management


In [73]:
#| include: false

## Check which categories have been dropped or added
old_categories = set(counts['job_category'].unique())
mapped_new_categories = set(new_counts['mapped_job_category'].unique())

mapped_added_categories = mapped_new_categories - old_categories
mapped_dropped_categories = old_categories - mapped_new_categories

mapped_unchanged = old_categories & mapped_new_categories

In [74]:
#| echo: false
new_counts_for_graph = new_counts.copy()
new_counts["job_category"] = new_counts["mapped_job_category"].copy()


In [75]:
#| include: false

# Compare the latest counts vs June 2022

latest_2026_date = new_counts["date"].tolist()[-1]
latest_2026_counts = new_counts[new_counts["date"] == latest_2026_date].drop("mapped_job_category", axis=1)
latest_2026_counts["count"].sum()
combined_table["June 2022"].sum()
combined_table["July 2023"].sum()
combined_table["July 2023"].sum() / combined_table["June 2022"].sum()
latest_2026_counts["count"].sum() / combined_table["June 2022"].sum()
latest_2026_counts.columns = ["Job Category", "Count", "Date", "Updated Job Category"]
latest_2026_counts.head()

np.int64(19835)

np.int64(91205)

np.int64(8412)

np.float64(0.09223178553807357)

np.float64(0.21747711200043857)

,Job Category,Count,Date,Updated Job Category
1360,Administrative Support,563,2026-11-05,Administrative Support
1361,Machine Learning Science,501,2026-11-05,Applied Science
1362,Audio / Video / Photography Production,20,2026-11-05,"Audio, Video, and Photography Production"
1363,Business & Merchant Development,295,2026-11-05,Business and Merchant Development
1364,Business Intelligence,202,2026-11-05,Business Intelligence and Data Engineering


In [76]:
#| include: false
 
latest_comparison = combined_table[["Job Category", "June 2022", "July 2023", "% Change"]].merge(
    latest_2026_counts[["Job Category", "Count", 
                        "Updated Job Category"
                        ]],
    on="Job Category",
    how="left"
).rename(columns={"Count": "Nov 2026", "% Change": "% Change 2023 vs 2022"})
latest_comparison["Change 2026 vs 2022"] = latest_comparison["Nov 2026"] - latest_comparison["June 2022"]
latest_comparison["% Change 2026 vs 2022"] = round((latest_comparison["Change 2026 vs 2022"] / latest_comparison["June 2022"]) * 100, 1)


# latest_comparison.sort_values(by="% Change vs 2022", ascending=True)
selected_columns = ["Job Category", "June 2022", "July 2023", "Nov 2026", "% Change 2023 vs 2022", "% Change 2026 vs 2022",
                     "Updated Category Labels"]


## Open Position Counts in November 2026 {#2026-update}

The number of open positions at Amazon as of November 2026 shows a decline of 78% compared to the number of positions available in June 2022. This is a large decline but represents an improvement compared with July 2023, when the measured decline was 91%.

As of November 2026, only Fulfillment roles have shown growth in job openings since June 2022. This growth would appear to make sense in the support of maintenance and growth in Amazon retail revenues.

Facilities, Maintenance and Real Estate shows a decline of just 11% since June 2022 and a recovery of more than 100% since July 2023. An inspection of the available roles reveals the need for expertise and experience in hands-on maintenance and repair of physical equipment. This may make this category more resilient against the rollout of AI compared to job categories which have been heavily impacted, such as Software Development and Business Intelligence.

In [77]:
#| echo: false

column_map = {
    "Job Category": "Job Category",
    "June 2022": "June 2022",
    "July 2023": "July 2023",
    "Nov 2026": "Nov 2026",
    "% Change 2023 vs 2022": "% Change 2023 vs 2022",
    "% Change 2026 vs 2022": "% Change 2026 vs 2022",
    "Updated Job Category": "Updated Category Labels",
}

plotly_table_df = (
    latest_comparison[list(column_map.keys())]
    .sort_values("% Change 2026 vs 2022", ascending=True)
    .rename(columns=column_map)
    .reset_index(drop=True)
)

fig_table = go.Figure(
    go.Table(
        header=dict(
            values=list(plotly_table_df.columns),
            fill_color="#2c3e50",
            font=dict(color="white", size=12),
            align="center",
        ),
        cells=dict(
            values=[plotly_table_df[col] for col in plotly_table_df.columns],
            fill_color=[["#f7f7f7", "white"] * (len(plotly_table_df) // 2 + 1)],
            align="center",
            height=26,
        ),
    )
)
fig_table.update_layout(
    title=dict(
        text="Amazon Job Counts - Updated Nov 2026",
        x=0.5,
        xanchor="center"
    ),
    template="plotly_white",
    height=1000,
    margin={"t": 60, "l": 10, "r": 10, "b": 10},
    
)


In [78]:
#| echo: false
build_category_dropdown_figure(new_counts_for_graph, date_range="Sept 2023 - Nov 2023")